<a href="https://colab.research.google.com/github/CalculatedContent/xgboost2ww/blob/main/notebooks/SpamBase_Hyperparameter_Sweep_Alpha2_ManifoldSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Introduction

This repaired notebook updates the SpamBase alpha search so it is **sweep-aware and evidence-driven** rather than globally forcing raw W7 alpha near 2.

Key empirical lesson:
- alpha on SpamBase is **sweep-local** (relationship depends on the hyperparameter family), not globally monotone across all sweeps.
- lowering alpha by reducing useful model capacity (for example, overly shallow trees) is not the same as lowering alpha while preserving representation quality.
- the workflow is now **manifold-based**: first identify sweep families that lower alpha without harming validation/test performance, then refine only along those families.

Raw W7 remains the primary matrix for cross-model comparison, and secondary matrix variants are evaluated later only on the final shortlist.

# 2. Reproduce Current Sweep Results

We keep the original dataset loading, split protocol, baseline candidate selection, local sweeps, model training utilities, xgboost2ww conversion path, and WeightWatcher extraction path.


## Google Drive checkpoint + local Mac compatibility

This cell keeps a Colab Google Drive checkpoint path and a local fallback path so the notebook runs both in Colab and locally on macOS/Linux.

In [ ]:

from pathlib import Path
import os

IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CHECKPOINT_ROOT_COLAB = Path('/content/drive/MyDrive/xgboost2ww_checkpoints')
CHECKPOINT_ROOT_LOCAL = Path('./checkpoints')
RESULTS_ROOT_LOCAL = Path('./results')
EXPERIMENT_NAME = 'spambase_alpha_trend_sweep'

if IN_COLAB:
    drive.mount('/content/drive')
    CHECKPOINT_ROOT = CHECKPOINT_ROOT_COLAB
else:
    CHECKPOINT_ROOT = CHECKPOINT_ROOT_LOCAL

CHECKPOINT_DIR = CHECKPOINT_ROOT / EXPERIMENT_NAME
RESULTS_DIR = RESULTS_ROOT_LOCAL / EXPERIMENT_NAME
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('RESULTS_DIR:', RESULTS_DIR)


In [ ]:
# Optional install cell (for Colab)
# !pip -q install xgboost weightwatcher xgboost2ww scikit-learn pandas matplotlib seaborn scipy


## Imports and global settings

In [ ]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

from scipy.stats import spearmanr, pearsonr

import xgboost as xgb
import weightwatcher as ww
from xgboost2ww import convert


In [ ]:

RNG = 123
np.random.seed(RNG)

TEST_SIZE = 0.20
VALID_SIZE_FROM_TRAIN = 0.20
BASELINE_SEEDS = [11, 29, 47]

W_MATRIX_PRIMARY = 'W7'
W_MATRIX_SECONDARY = 'W8'  # optional
RUN_SECONDARY_W8 = False   # set True if runtime budget allows

WW_ANALYZE_OPTIONS = dict(randomize=True, detX=True, fix_fingers='clip_xmax')
WW_MIN_TAIL = 10

BASELINE_NAME = 'spambase_hist_baseline_v1'

BASE_OBJECTIVE_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': ['logloss', 'auc'],
    'tree_method': 'hist'
}


## Load SpamBase with deterministic fallback and fixed preprocessing

In [ ]:

SPAMBASE_FEATURES = [
    'word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d', 'word_freq_our', 'word_freq_over',
    'word_freq_remove', 'word_freq_internet', 'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will',
    'word_freq_people', 'word_freq_report', 'word_freq_addresses', 'word_freq_free', 'word_freq_business', 'word_freq_email',
    'word_freq_you', 'word_freq_credit', 'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money',
    'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_650', 'word_freq_lab', 'word_freq_labs',
    'word_freq_telnet', 'word_freq_857', 'word_freq_data', 'word_freq_415', 'word_freq_85', 'word_freq_technology',
    'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct', 'word_freq_cs', 'word_freq_meeting',
    'word_freq_original', 'word_freq_project', 'word_freq_re', 'word_freq_edu', 'word_freq_table', 'word_freq_conference',
    'char_freq_;', 'char_freq_(', 'char_freq_[', 'char_freq_!', 'char_freq_$', 'char_freq_#',
    'capital_run_length_average', 'capital_run_length_longest', 'capital_run_length_total'
]


def load_spambase_data():
    try:
        ds = fetch_openml(name='spambase', version=1, as_frame=False, parser='auto')
        X = ds.data.astype(np.float32)
        y = np.asarray(ds.target)
        if y.dtype.kind in 'OUS':
            y = np.where(np.isin(y.astype(str), ['1', 'spam', 'yes', 'True']), 1, 0)
        y = y.astype(int)
        source = 'openml'
    except Exception as e:
        print(f'OpenML fetch failed ({type(e).__name__}: {e}); using UCI fallback CSV.')
        df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data', header=None)
        X = df.iloc[:, :-1].to_numpy(dtype=np.float32)
        y = df.iloc[:, -1].to_numpy(dtype=int)
        source = 'uci_csv'

    if np.isnan(X).any():
        med = np.nanmedian(X, axis=0)
        inds = np.where(np.isnan(X))
        X[inds] = med[inds[1]]

    return X, y, source

X, y, data_source = load_spambase_data()

idx_train_full, idx_test = train_test_split(
    np.arange(len(y)), test_size=TEST_SIZE, random_state=RNG, stratify=y
)
idx_train, idx_valid = train_test_split(
    idx_train_full, test_size=VALID_SIZE_FROM_TRAIN, random_state=RNG, stratify=y[idx_train_full]
)

X_train, y_train = X[idx_train], y[idx_train]
X_valid, y_valid = X[idx_valid], y[idx_valid]
X_test, y_test = X[idx_test], y[idx_test]
X_train_valid, y_train_valid = X[idx_train_full], y[idx_train_full]

print('Data source:', data_source)
print('Shapes train/valid/test:', X_train.shape, X_valid.shape, X_test.shape)
print('Class balance train/valid/test:', y_train.mean(), y_valid.mean(), y_test.mean())


## Baseline anchor model

The baseline is the anchor point for all sweeps: it is selected by validation performance and every sweep perturbs only one mechanism from this fixed reference.

In [ ]:

BASELINE_CANDIDATES = [
    dict(max_depth=5, eta=0.08, n_estimators=1600, min_child_weight=3, subsample=0.85, colsample_bytree=0.85, gamma=0.0, reg_alpha=0.0, reg_lambda=1.0),
    dict(max_depth=6, eta=0.06, n_estimators=2200, min_child_weight=3, subsample=0.90, colsample_bytree=0.90, gamma=0.0, reg_alpha=0.0, reg_lambda=1.0),
    dict(max_depth=4, eta=0.10, n_estimators=1200, min_child_weight=5, subsample=0.80, colsample_bytree=0.85, gamma=0.0, reg_alpha=0.0, reg_lambda=2.0),
]


def train_single_config(params, seed):
    xgb_params = {**BASE_OBJECTIVE_PARAMS,
                  'seed': int(seed),
                  'max_depth': int(params['max_depth']),
                  'eta': float(params['eta']),
                  'subsample': float(params['subsample']),
                  'colsample_bytree': float(params['colsample_bytree']),
                  'min_child_weight': float(params['min_child_weight']),
                  'gamma': float(params['gamma']),
                  'reg_alpha': float(params['reg_alpha']),
                  'reg_lambda': float(params['reg_lambda'])}

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    dtrain_valid = xgb.DMatrix(X_train_valid, label=y_train_valid)
    dtest = xgb.DMatrix(X_test, label=y_test)

    model = xgb.train(
        params=xgb_params,
        dtrain=dtrain,
        num_boost_round=int(params['n_estimators']),
        evals=[(dtrain, 'train'), (dvalid, 'valid')],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    best_round = int(getattr(model, 'best_iteration', int(params['n_estimators']) - 1) + 1)

    p_train = model.predict(dtrain, iteration_range=(0, best_round))
    p_valid = model.predict(dvalid, iteration_range=(0, best_round))
    p_test = model.predict(dtest, iteration_range=(0, best_round))

    metrics = dict(
        train_accuracy=accuracy_score(y_train, (p_train >= 0.5).astype(int)),
        valid_accuracy=accuracy_score(y_valid, (p_valid >= 0.5).astype(int)),
        test_accuracy=accuracy_score(y_test, (p_test >= 0.5).astype(int)),
        train_auc=roc_auc_score(y_train, p_train),
        valid_auc=roc_auc_score(y_valid, p_valid),
        test_auc=roc_auc_score(y_test, p_test),
        train_logloss=log_loss(y_train, p_train),
        valid_logloss=log_loss(y_valid, p_valid),
        test_logloss=log_loss(y_test, p_test),
        best_round=best_round,
    )

    return model, metrics, xgb_params


def build_baseline_params():
    rows = []
    for i, cand in enumerate(BASELINE_CANDIDATES):
        valid_scores = []
        for seed in BASELINE_SEEDS:
            _, m, _ = train_single_config(cand, seed)
            valid_scores.append(m['valid_accuracy'])
        rows.append({'candidate_id': i, 'params': cand, 'valid_mean': float(np.mean(valid_scores)), 'valid_std': float(np.std(valid_scores))})

    baseline_df = pd.DataFrame(rows).sort_values(['valid_mean', 'valid_std'], ascending=[False, True]).reset_index(drop=True)
    return baseline_df.iloc[0]['params'], baseline_df

BASELINE_PARAMS, baseline_selection_df = build_baseline_params()
print('BASELINE_NAME:', BASELINE_NAME)
print('BASELINE_SEEDS:', BASELINE_SEEDS)
print('BASELINE_PARAMS:', BASELINE_PARAMS)
baseline_selection_df


## Controlled local sweep grids

Each sweep changes one structural mechanism with all other parameters fixed at `BASELINE_PARAMS`.

In [ ]:
def _unique_sorted(values, round_to=6):
    vals = sorted(set([round(float(v), round_to) for v in values]))
    return vals


def build_local_sweeps(baseline_params):
    b = baseline_params

    n_base = int(b['n_estimators'])
    n_grid = sorted(set(max(100, int(round(n_base * s))) for s in [0.35, 0.5, 0.75, 1.0, 1.3, 1.6, 2.0]))

    d_base = int(b['max_depth'])
    depth_grid = sorted(set(int(np.clip(d_base + d, 2, 10)) for d in [-3, -2, -1, 0, 1]))

    mcw_base = float(b['min_child_weight'])
    mcw_grid = sorted(set(max(1.0, round(mcw_base * s, 3)) for s in [0.5, 1.0, 2.0, 4.0, 8.0]))

    s_base = float(b['subsample'])
    subsample_grid = _unique_sorted([
        max(0.5, s_base - 0.30),
        max(0.6, s_base - 0.20),
        max(0.7, s_base - 0.10),
        s_base,
        min(1.0, s_base + 0.05),
    ], round_to=3)

    eta_base = float(b['eta'])
    eta_grid = sorted(set(float(np.clip(eta_base * s, 0.015, 0.3)) for s in [0.4, 0.65, 1.0, 1.5]))
    eta_n_pairs = []
    constant_budget = eta_base * n_base
    for eta in eta_grid:
        n = int(max(150, round(constant_budget / eta)))
        eta_n_pairs.append((eta, n))

    return {
        'n_estimators': n_grid,
        'max_depth': depth_grid,
        'min_child_weight': mcw_grid,
        'subsample': subsample_grid,
        'eta_x_n_estimators': eta_n_pairs,
    }

LOCAL_SWEEPS = build_local_sweeps(BASELINE_PARAMS)
LOCAL_SWEEPS


## Training loop (fixed protocol)

Purpose: evaluate each configuration with the same split, same metric stack, and same repeated seeds before WeightWatcher extraction.

In [ ]:
def extract_weightwatcher_metrics(details_df, matrix_kind):
    out = {
        'matrix_kind': matrix_kind,
        'alpha_primary': np.nan,
        'alpha_mean': np.nan,
        'alpha_min': np.nan,
        'alpha_q25': np.nan,
        'n_valid_layers': 0,
        'number_of_layers_analyzed': 0,
        'ww_fit_success': False,
        'effective_rank': np.nan,
        'rank': np.nan,
        'trace': np.nan,
        'frobenius_norm': np.nan,
        'n_rows': np.nan,
        'n_cols': np.nan,
    }

    if not isinstance(details_df, pd.DataFrame) or details_df.empty:
        return out

    out['number_of_layers_analyzed'] = int(len(details_df))

    alpha_col = 'alpha' if 'alpha' in details_df.columns else None
    if alpha_col is not None:
        alpha_vals = pd.to_numeric(details_df[alpha_col], errors='coerce')
        tail_col = None
        for c in ['num_evals', 'num_pl_spikes', 'M']:
            if c in details_df.columns:
                tail_col = c
                break
        if tail_col is not None:
            tail_vals = pd.to_numeric(details_df[tail_col], errors='coerce')
            valid_mask = alpha_vals.notna() & np.isfinite(alpha_vals) & (tail_vals.fillna(WW_MIN_TAIL) >= WW_MIN_TAIL)
        else:
            valid_mask = alpha_vals.notna() & np.isfinite(alpha_vals)

        valid_alphas = alpha_vals[valid_mask]
        out['n_valid_layers'] = int(valid_alphas.shape[0])
        if out['n_valid_layers'] > 0:
            out['alpha_q25'] = float(np.quantile(valid_alphas, 0.25))
            out['alpha_min'] = float(np.min(valid_alphas))
            out['alpha_mean'] = float(np.mean(valid_alphas))
            # Use low-tail alpha as primary signal to better capture approach-to-2 behavior.
            out['alpha_primary'] = float(out['alpha_q25'])
            out['ww_fit_success'] = True

    for c in ['effective_rank', 'erank']:
        if c in details_df.columns:
            out['effective_rank'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break
    for c in ['rank']:
        if c in details_df.columns:
            out['rank'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break
    for c in ['trace', 'matrix_trace']:
        if c in details_df.columns:
            out['trace'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break
    for c in ['fro', 'frobenius_norm', 'norm']:
        if c in details_df.columns:
            out['frobenius_norm'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break
    for c in ['N', 'n_rows']:
        if c in details_df.columns:
            out['n_rows'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break
    for c in ['M', 'n_cols']:
        if c in details_df.columns:
            out['n_cols'] = float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
            break

    return out


def run_config_across_seeds(sweep_name, sweep_value, params):
    run_rows, layer_rows = [], []
    for seed in BASELINE_SEEDS:
        model, metrics, train_params = train_single_config(params, seed)

        converted = convert(
            model=model,
            data=X_train_valid,
            labels=y_train_valid,
            W=W_MATRIX_PRIMARY,
            nfolds=5,
            t_points=40,
            random_state=RNG,
            train_params=train_params,
            num_boost_round=metrics['best_round'],
            multiclass='error',
            return_type='torch',
            verbose=False,
        )
        details = ww.WeightWatcher(model=converted).analyze(**WW_ANALYZE_OPTIONS)
        ww_metrics = extract_weightwatcher_metrics(details, W_MATRIX_PRIMARY)

        config_id = f"{sweep_name}::{sweep_value}"
        row = {
            'sweep_name': sweep_name,
            'sweep_value': sweep_value,
            'seed': seed,
            'configuration_id': config_id,
            'is_baseline': bool(sweep_name == 'baseline'),
            **params,
            **metrics,
            **ww_metrics,
        }
        row['train_test_gap'] = row['train_accuracy'] - row['test_accuracy']
        run_rows.append(row)

        if isinstance(details, pd.DataFrame) and not details.empty:
            d = details.copy()
            d['sweep_name'] = sweep_name
            d['sweep_value'] = sweep_value
            d['seed'] = seed
            d['configuration_id'] = config_id
            layer_rows.append(d)

    return run_rows, layer_rows


def aggregate_results(per_seed_df):
    metric_cols = ['train_accuracy', 'valid_accuracy', 'test_accuracy', 'alpha_primary', 'alpha_q25', 'alpha_mean', 'alpha_min', 'effective_rank', 'train_test_gap']
    agg_dict = {c: ['mean', 'std'] for c in metric_cols}
    group_cols = ['sweep_name', 'sweep_value', 'configuration_id', 'is_baseline']
    agg_df = per_seed_df.groupby(group_cols, as_index=False).agg(agg_dict)
    agg_df.columns = ['_'.join(c).strip('_') for c in agg_df.columns]
    return agg_df


In [ ]:
def build_params_from_baseline(base, key, value):
    p = dict(base)
    p[key] = value
    return p


def build_depth_compensated_params(base, depth_value):
    p = dict(base)
    p['max_depth'] = int(depth_value)
    d_base = float(base['max_depth'])
    n_base = int(base['n_estimators'])
    depth_scale = (d_base / max(2.0, float(depth_value))) ** 1.15
    p['n_estimators'] = int(np.clip(round(n_base * depth_scale), 200, 6000))
    return p


experiment_plan = []

# Baseline row
experiment_plan.append(dict(sweep_name='baseline', sweep_value='baseline', params=dict(BASELINE_PARAMS)))

# Primary sweeps
for v in LOCAL_SWEEPS['n_estimators']:
    experiment_plan.append(dict(sweep_name='n_estimators', sweep_value=v, params=build_params_from_baseline(BASELINE_PARAMS, 'n_estimators', int(v))))
for v in LOCAL_SWEEPS['max_depth']:
    # Compensate shallower trees with more boosting rounds to reduce underfit-driven reversals.
    experiment_plan.append(dict(sweep_name='max_depth', sweep_value=v, params=build_depth_compensated_params(BASELINE_PARAMS, int(v))))
for v in LOCAL_SWEEPS['min_child_weight']:
    experiment_plan.append(dict(sweep_name='min_child_weight', sweep_value=v, params=build_params_from_baseline(BASELINE_PARAMS, 'min_child_weight', float(v))))
for v in LOCAL_SWEEPS['subsample']:
    experiment_plan.append(dict(sweep_name='subsample', sweep_value=v, params=build_params_from_baseline(BASELINE_PARAMS, 'subsample', float(v))))

# Optional structured two-parameter sweep
for eta, n_estimators in LOCAL_SWEEPS['eta_x_n_estimators']:
    p = dict(BASELINE_PARAMS)
    p['eta'] = float(eta)
    p['n_estimators'] = int(n_estimators)
    experiment_plan.append(dict(sweep_name='eta_x_n_estimators', sweep_value=f'eta={eta:.4f}|n={n_estimators}', params=p))

len(experiment_plan)


## WeightWatcher extraction

Purpose: run one fixed WW pipeline (`W7`, same analyze options) for every trained model and extract standardized alpha and matrix metadata.

In [ ]:

all_rows = []
all_layer_rows = []

for item in experiment_plan:
    if (item['sweep_name'] == 'eta_x_n_estimators') and (not RUN_SECONDARY_W8):
        # still run this sweep because it is part of the design; W8 remains optional
        pass
    rows, layer_rows = run_config_across_seeds(item['sweep_name'], item['sweep_value'], item['params'])
    all_rows.extend(rows)
    all_layer_rows.extend(layer_rows)

per_seed_results_df = pd.DataFrame(all_rows)
layer_long_df = pd.concat(all_layer_rows, ignore_index=True) if all_layer_rows else pd.DataFrame()
aggregated_results_df = aggregate_results(per_seed_results_df)

print('Per-seed rows:', per_seed_results_df.shape)
print('Aggregated rows:', aggregated_results_df.shape)
print('Layer-long rows:', layer_long_df.shape)
per_seed_results_df.head()


# 3. Sweep-Local Alpha–Accuracy Diagnostics

In [ ]:

# Rebuild aggregated table with richer metadata and explicit naming used by the manifold workflow.
PARAM_COLS = ['max_depth','eta','n_estimators','min_child_weight','subsample','colsample_bytree','gamma','reg_alpha','reg_lambda']

for df in [per_seed_results_df, aggregated_results_df]:
    if 'valid_accuracy_mean' not in df.columns and 'valid_accuracy' in df.columns:
        pass

if 'train_test_gap' not in per_seed_results_df.columns:
    per_seed_results_df['train_test_gap'] = per_seed_results_df['train_accuracy'] - per_seed_results_df['test_accuracy']
per_seed_results_df['train_val_gap'] = per_seed_results_df['train_accuracy'] - per_seed_results_df['valid_accuracy']
per_seed_results_df['alpha_distance_to_2'] = (per_seed_results_df['alpha_primary'] - 2.0).abs()

agg_metric_cols = ['train_accuracy','valid_accuracy','test_accuracy','alpha_primary','alpha_distance_to_2','train_val_gap','train_test_gap','effective_rank']
agg_dict = {c:['mean','std'] for c in agg_metric_cols if c in per_seed_results_df.columns}
aggregated_results_df = per_seed_results_df.groupby(['configuration_id','sweep_name','sweep_value'], as_index=False).agg(agg_dict)
aggregated_results_df.columns = ['_'.join(c).strip('_') for c in aggregated_results_df.columns]

param_view = per_seed_results_df.groupby('configuration_id', as_index=False)[PARAM_COLS].first()
meta_view = per_seed_results_df.groupby('configuration_id', as_index=False).agg(is_baseline=('is_baseline','max'))
aggregated_results_df = aggregated_results_df.merge(param_view, on='configuration_id', how='left').merge(meta_view, on='configuration_id', how='left')
aggregated_results_df['search_stage'] = np.where(aggregated_results_df['is_baseline'], 'current_sweep', 'current_sweep')
aggregated_results_df['anchor_name'] = np.where(aggregated_results_df['is_baseline'], 'ACCURACY_BASELINE', '')
aggregated_results_df['hyperparameters'] = aggregated_results_df[PARAM_COLS].apply(lambda r: str({k:(int(v) if k in ['max_depth','n_estimators'] and pd.notna(v) else float(v) if pd.notna(v) else None) for k,v in r.items()}), axis=1)

ACCURACY_BASELINE = aggregated_results_df.sort_values('valid_accuracy_mean', ascending=False).iloc[0].copy()
print('ACCURACY_BASELINE configuration_id:', ACCURACY_BASELINE['configuration_id'])
print('ACCURACY_BASELINE val/test/alpha:', ACCURACY_BASELINE['valid_accuracy_mean'], ACCURACY_BASELINE['test_accuracy_mean'], ACCURACY_BASELINE['alpha_primary_mean'])

def safe_corr(x, y):
    x = pd.to_numeric(pd.Series(x), errors='coerce')
    y = pd.to_numeric(pd.Series(y), errors='coerce')
    m = x.notna() & y.notna()
    if m.sum() < 3:
        return np.nan
    return float(spearmanr(x[m], y[m])[0])

def safe_slope(x, y):
    x = pd.to_numeric(pd.Series(x), errors='coerce')
    y = pd.to_numeric(pd.Series(y), errors='coerce')
    m = x.notna() & y.notna()
    if m.sum() < 2:
        return np.nan
    return float(np.polyfit(x[m], y[m], 1)[0])

diag_rows = []
for sweep, sdf in aggregated_results_df.groupby('sweep_name'):
    alpha_range = float(sdf['alpha_primary_mean'].max() - sdf['alpha_primary_mean'].min())
    sp_val = safe_corr(sdf['alpha_primary_mean'], sdf['valid_accuracy_mean'])
    sp_test = safe_corr(sdf['alpha_primary_mean'], sdf['test_accuracy_mean'])
    if alpha_range >= 0.15 and ((pd.notna(sp_test) and sp_test <= -0.2) or (pd.notna(sp_val) and sp_val <= -0.2)):
        label = 'alpha_helpful'
    elif alpha_range >= 0.15 and ((pd.notna(sp_test) and sp_test >= 0.2) or (pd.notna(sp_val) and sp_val >= 0.2)):
        label = 'capacity_tradeoff'
    else:
        label = 'inconclusive'
    diag_rows.append({
        'sweep_name': sweep,
        'number_of_configurations': int(len(sdf)),
        'alpha_range': alpha_range,
        'val_accuracy_range': float(sdf['valid_accuracy_mean'].max() - sdf['valid_accuracy_mean'].min()),
        'test_accuracy_range': float(sdf['test_accuracy_mean'].max() - sdf['test_accuracy_mean'].min()),
        'spearman_alpha_val': sp_val,
        'spearman_alpha_test': sp_test,
        'spearman_alpha_train_test_gap': safe_corr(sdf['alpha_primary_mean'], sdf['train_test_gap_mean']),
        'slope_test_vs_alpha': safe_slope(sdf['alpha_primary_mean'], sdf['test_accuracy_mean']),
        'slope_val_vs_alpha': safe_slope(sdf['alpha_primary_mean'], sdf['valid_accuracy_mean']),
        'classification': label,
    })

sweep_diagnostics_df = pd.DataFrame(diag_rows).sort_values('sweep_name').reset_index(drop=True)
display(sweep_diagnostics_df)


### Why the first alpha=2 target failed under raw W7

Current evidence is interpreted **per sweep family**, not globally. If max_depth and/or eta×n_estimators show positive alpha–accuracy correlation, then lower alpha there is likely capacity loss rather than better generalization structure. Subsample is expected to be the most promising low-alpha direction so far.


# 4. Identify Helpful Search Manifolds

In [ ]:

HELPFUL_SWEEPS = sweep_diagnostics_df.loc[sweep_diagnostics_df['classification']=='alpha_helpful','sweep_name'].tolist()
if len(HELPFUL_SWEEPS) == 0:
    HELPFUL_SWEEPS = ['subsample']
    print('WARNING: no sweep met alpha_helpful rule; falling back to ["subsample"].')

CAPACITY_TRADEOFF_SWEEPS = sweep_diagnostics_df.loc[sweep_diagnostics_df['classification']=='capacity_tradeoff','sweep_name'].tolist()
INCONCLUSIVE_SWEEPS = sweep_diagnostics_df.loc[sweep_diagnostics_df['classification']=='inconclusive','sweep_name'].tolist()

print('HELPFUL_SWEEPS:', HELPFUL_SWEEPS)
print('CAPACITY_TRADEOFF_SWEEPS:', CAPACITY_TRADEOFF_SWEEPS)
print('INCONCLUSIVE_SWEEPS:', INCONCLUSIVE_SWEEPS)


# 5. Build the Low-Alpha Competitive Anchor

In [ ]:

helpful_df = aggregated_results_df[aggregated_results_df['sweep_name'].isin(HELPFUL_SWEEPS)].copy()
best_val = float(aggregated_results_df['valid_accuracy_mean'].max())
competitive_tol = 0.002
competitive_helpful_df = helpful_df[helpful_df['valid_accuracy_mean'] >= best_val - competitive_tol].copy()

if competitive_helpful_df.empty and not helpful_df.empty:
    thresh = helpful_df['valid_accuracy_mean'].quantile(0.8)
    competitive_helpful_df = helpful_df[helpful_df['valid_accuracy_mean'] >= thresh].copy()

if competitive_helpful_df.empty:
    LOW_ALPHA_COMPETITIVE_ANCHOR = ACCURACY_BASELINE.copy()
    print('No helpful-sweep rows available; LOW_ALPHA_COMPETITIVE_ANCHOR falls back to ACCURACY_BASELINE.')
else:
    LOW_ALPHA_COMPETITIVE_ANCHOR = competitive_helpful_df.sort_values(['alpha_primary_mean','valid_accuracy_mean'], ascending=[True,False]).iloc[0].copy()

aggregated_results_df.loc[aggregated_results_df['configuration_id']==LOW_ALPHA_COMPETITIVE_ANCHOR['configuration_id'],'anchor_name'] = 'LOW_ALPHA_COMPETITIVE_ANCHOR'

print('LOW_ALPHA_COMPETITIVE_ANCHOR:', LOW_ALPHA_COMPETITIVE_ANCHOR['configuration_id'])
print('val/test/alpha:', LOW_ALPHA_COMPETITIVE_ANCHOR['valid_accuracy_mean'], LOW_ALPHA_COMPETITIVE_ANCHOR['test_accuracy_mean'], LOW_ALPHA_COMPETITIVE_ANCHOR['alpha_primary_mean'])


### Why the new search is manifold-based

Targeted refinement now starts from `LOW_ALPHA_COMPETITIVE_ANCHOR` and follows only empirically helpful directions. Capacity-tradeoff sweeps remain diagnostic, not primary optimization axes.


# 6. Targeted Refinement Around the Helpful Manifold

In [ ]:

# Build conservative refinement candidates centered on LOW_ALPHA_COMPETITIVE_ANCHOR
anchor_params = {k: LOW_ALPHA_COMPETITIVE_ANCHOR[k] for k in PARAM_COLS if k in LOW_ALPHA_COMPETITIVE_ANCHOR.index}
anchor_depth = int(anchor_params['max_depth'])

max_depth_sweep = aggregated_results_df[aggregated_results_df['sweep_name']=='max_depth'].copy()
DEPTH_CANDIDATES = [anchor_depth]
if anchor_depth + 1 <= 12:
    DEPTH_CANDIDATES.append(anchor_depth + 1)
if not max_depth_sweep.empty and 'sweep_value' in max_depth_sweep.columns:
    top_depths = max_depth_sweep.sort_values('valid_accuracy_mean', ascending=False)['sweep_value'].astype(float).round().astype(int).head(2).tolist()
    if (anchor_depth - 1) in top_depths and (anchor_depth - 1) >= 2:
        DEPTH_CANDIDATES.append(anchor_depth - 1)
DEPTH_CANDIDATES = sorted(set(DEPTH_CANDIDATES))

eta_anchor = float(anchor_params['eta'])
eta_candidates = sorted(set(float(np.clip(v, 0.02, 0.2)) for v in [0.8*eta_anchor, eta_anchor, 1.2*eta_anchor]))
base_budget = eta_anchor * float(anchor_params['n_estimators'])
n_mult = [0.9, 1.0, 1.1, 1.25]
eta_n_candidates = []
for eta in eta_candidates:
    comp_n = max(100, int(round(base_budget / eta)))
    for m in n_mult:
        eta_n_candidates.append((eta, int(max(100, round(comp_n*m)))))
eta_n_candidates = sorted(set(eta_n_candidates))

subsample_candidates = _unique_sorted(np.clip([
    anchor_params['subsample'] - 0.10,
    anchor_params['subsample'] - 0.05,
    anchor_params['subsample'] - 0.025,
    anchor_params['subsample'],
    anchor_params['subsample'] + 0.025,
    anchor_params['subsample'] + 0.05,
    1.0
], 0.8, 1.0), round_to=3)

colsample_candidates = _unique_sorted(np.clip([
    anchor_params['colsample_bytree'] - 0.10,
    anchor_params['colsample_bytree'] - 0.05,
    anchor_params['colsample_bytree'],
    anchor_params['colsample_bytree'] + 0.05,
    1.0
], 0.8, 1.0), round_to=3)

mcw_candidates = sorted(set(int(np.clip(round(v),4,64)) for v in [anchor_params['min_child_weight'], 2*anchor_params['min_child_weight'], 4*anchor_params['min_child_weight']]))
gamma_candidates = [0.0,0.25,0.5,1.0,2.0]
lambda_candidates = [1.0,3.0,10.0,30.0]

axis_grids = {
    'subsample': subsample_candidates,
    'colsample_bytree': colsample_candidates,
    'min_child_weight': mcw_candidates,
    'gamma': gamma_candidates,
    'reg_lambda': lambda_candidates,
    'max_depth': DEPTH_CANDIDATES,
    'eta_x_n_estimators': eta_n_candidates,
}
if 'colsample_bynode' in per_seed_results_df.columns:
    axis_grids['colsample_bynode'] = _unique_sorted(np.clip([
        float(anchor_params.get('colsample_bynode', anchor_params['colsample_bytree'])) - 0.10,
        float(anchor_params.get('colsample_bynode', anchor_params['colsample_bytree'])) - 0.05,
        float(anchor_params.get('colsample_bynode', anchor_params['colsample_bytree'])),
        float(anchor_params.get('colsample_bynode', anchor_params['colsample_bytree'])) + 0.05,
        1.0
    ], 0.8, 1.0), round_to=3)

phase_a_plan = []
for axis, values in axis_grids.items():
    for v in values:
        p = dict(anchor_params)
        if axis == 'eta_x_n_estimators':
            eta, ne = v
            p['eta'] = float(eta)
            p['n_estimators'] = int(ne)
            sweep_value = f'eta={eta:.4f}|n={ne}'
        else:
            p[axis] = int(v) if axis in ['max_depth','n_estimators','min_child_weight'] else float(v)
            sweep_value = v
        phase_a_plan.append(dict(sweep_name=f'refine_A_{axis}', sweep_value=sweep_value, params=p))

phase_a_rows, phase_a_layers = [], []
for item in phase_a_plan:
    rows, lrows = run_config_across_seeds(item['sweep_name'], item['sweep_value'], item['params'])
    phase_a_rows.extend(rows)
    phase_a_layers.extend(lrows)

phase_a_df = pd.DataFrame(phase_a_rows)

# Pick top settings per axis, then small joint phase-B combinations.
axis_best_params = {}
if not phase_a_df.empty:
    aagg = phase_a_df.groupby(['sweep_name','configuration_id'], as_index=False).agg(valid_accuracy=('valid_accuracy','mean'), alpha=('alpha_primary','mean'))
    for sweep_name, sdf in aagg.groupby('sweep_name'):
        top_cfg = sdf.sort_values(['valid_accuracy','alpha'], ascending=[False,True]).head(3)['configuration_id'].tolist()
        axis_best_params[sweep_name] = top_cfg

phase_b_seed = dict(anchor_params)
phase_b_candidates = []
for cfgs in axis_best_params.values():
    for cfg in cfgs[:2]:
        p = phase_a_df[phase_a_df['configuration_id']==cfg].iloc[0][PARAM_COLS].to_dict()
        phase_b_candidates.append(p)

# dedupe and cap
uniq = []
seen = set()
for p in phase_b_candidates:
    key = tuple((k, round(float(p[k]),6) if k not in ['max_depth','n_estimators'] else int(p[k])) for k in PARAM_COLS)
    if key not in seen:
        seen.add(key)
        uniq.append(p)
phase_b_candidates = uniq[:24]

phase_b_rows, phase_b_layers = [], []
for i, p in enumerate(phase_b_candidates):
    rows, lrows = run_config_across_seeds('refine_B_joint', f'joint_{i:02d}', p)
    phase_b_rows.extend(rows)
    phase_b_layers.extend(lrows)

refine_rows = pd.DataFrame(phase_a_rows + phase_b_rows)
if not refine_rows.empty:
    refine_rows['search_stage'] = np.where(refine_rows['sweep_name'].str.startswith('refine_A'), 'targeted_refine_A', 'targeted_refine_B')
    refine_rows['anchor_name'] = 'LOW_ALPHA_COMPETITIVE_ANCHOR'

per_seed_results_df = pd.concat([per_seed_results_df, refine_rows], ignore_index=True)
if phase_a_layers or phase_b_layers:
    layer_long_df = pd.concat([layer_long_df, pd.concat(phase_a_layers + phase_b_layers, ignore_index=True)], ignore_index=True)

# recompute aggregate after refinement
per_seed_results_df['train_val_gap'] = per_seed_results_df['train_accuracy'] - per_seed_results_df['valid_accuracy']
per_seed_results_df['train_test_gap'] = per_seed_results_df['train_accuracy'] - per_seed_results_df['test_accuracy']
per_seed_results_df['alpha_distance_to_2'] = (per_seed_results_df['alpha_primary'] - 2.0).abs()

agg_dict = {c:['mean','std'] for c in agg_metric_cols if c in per_seed_results_df.columns}
aggregated_results_df = per_seed_results_df.groupby(['configuration_id','sweep_name','sweep_value'], as_index=False).agg(agg_dict)
aggregated_results_df.columns = ['_'.join(c).strip('_') for c in aggregated_results_df.columns]
param_view = per_seed_results_df.groupby('configuration_id', as_index=False)[PARAM_COLS].first()
meta_view = per_seed_results_df.groupby('configuration_id', as_index=False).agg(
    is_baseline=('is_baseline','max'),
    search_stage=('search_stage', lambda s: s.dropna().iloc[-1] if s.dropna().shape[0] else 'current_sweep'),
    anchor_name=('anchor_name', lambda s: s.dropna().iloc[-1] if s.dropna().shape[0] else ''),
)
aggregated_results_df = aggregated_results_df.merge(param_view, on='configuration_id', how='left').merge(meta_view, on='configuration_id', how='left')
aggregated_results_df['hyperparameters'] = aggregated_results_df[PARAM_COLS].apply(lambda r: str({k:(int(v) if k in ['max_depth','n_estimators'] and pd.notna(v) else float(v) if pd.notna(v) else None) for k,v in r.items()}), axis=1)
print('Total aggregated rows after targeted refinement:', len(aggregated_results_df))


# 7. Raw W7 Frontier Analysis

In [ ]:

best_val_overall = float(aggregated_results_df['valid_accuracy_mean'].max())
competitive_tol = 0.002
competitive_raw_df = aggregated_results_df[aggregated_results_df['valid_accuracy_mean'] >= best_val_overall - competitive_tol].copy()
raw_w7_alpha_floor = float(competitive_raw_df['alpha_primary_mean'].min())
RAW_W7_ALPHA2_REACHABLE = bool(raw_w7_alpha_floor <= 2.3)

print('best_val_overall:', best_val_overall)
print('raw_w7_alpha_floor:', raw_w7_alpha_floor)
print('RAW_W7_ALPHA2_REACHABLE:', RAW_W7_ALPHA2_REACHABLE)
if not RAW_W7_ALPHA2_REACHABLE:
    print('Under the current raw W7 definition on SpamBase, alpha near 2.0 is not competitively reachable in the present search region.')

# Pareto frontier helpers

def pareto_frontier(df, x_col, y_col):
    d = df[[x_col,y_col,'configuration_id']].dropna().sort_values([x_col, y_col], ascending=[True, False])
    frontier = []
    best_y = -np.inf
    for _, r in d.iterrows():
        if r[y_col] > best_y + 1e-12:
            frontier.append(r['configuration_id'])
            best_y = r[y_col]
    return df[df['configuration_id'].isin(frontier)].copy()

pareto_alpha_df = pareto_frontier(aggregated_results_df, 'alpha_primary_mean', 'valid_accuracy_mean')
pareto_d2_df = pareto_frontier(aggregated_results_df, 'alpha_distance_to_2_mean', 'valid_accuracy_mean')

best_raw_w7_accuracy_model = aggregated_results_df.sort_values('valid_accuracy_mean', ascending=False).iloc[0].copy()
best_raw_w7_low_alpha_model = competitive_raw_df.sort_values(['alpha_primary_mean','valid_accuracy_mean'], ascending=[True,False]).iloc[0].copy()
pareto_alpha_df['raw_tradeoff_score'] = pareto_alpha_df['valid_accuracy_mean'] - 0.002 * np.maximum(0.0, pareto_alpha_df['alpha_primary_mean'] - raw_w7_alpha_floor)
best_raw_w7_pareto_model = pareto_alpha_df.sort_values('raw_tradeoff_score', ascending=False).iloc[0].copy()


In [ ]:

# Per-sweep plots with alpha and alpha-distance views
sweeps_to_plot = sorted(aggregated_results_df['sweep_name'].unique())
for sweep in sweeps_to_plot:
    sdf = aggregated_results_df[aggregated_results_df['sweep_name']==sweep].copy()
    if sdf.empty:
        continue
    fig, axes = plt.subplots(1, 5, figsize=(24,4))
    x1 = 'alpha_primary_mean'; x2 = 'alpha_distance_to_2_mean'
    pairs = [
        (x1,'valid_accuracy_mean','val acc vs alpha_primary'),
        (x1,'test_accuracy_mean','test acc vs alpha_primary'),
        (x1,'train_test_gap_mean','train-test gap vs alpha_primary'),
        (x2,'valid_accuracy_mean','val acc vs |alpha-2|'),
        (x2,'test_accuracy_mean','test acc vs |alpha-2|'),
    ]
    for ax,(x,y,title) in zip(axes,pairs):
        ax.scatter(sdf[x], sdf[y], s=60)
        for _, r in sdf.iterrows():
            ax.annotate(str(r['sweep_value']), (r[x], r[y]), fontsize=7)
        if x == 'alpha_primary_mean':
            ax.axvline(2.0, color='red', linestyle='--', linewidth=1)
        ax.set_title(f'{sweep}: {title}')
        ax.set_xlabel(x)
        ax.set_ylabel(y)
    plt.tight_layout(); plt.show()

# Overall diagnostics and frontier plots
fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(data=sweep_diagnostics_df, x='sweep_name', y='spearman_alpha_test', hue='classification', ax=ax)
ax.axhline(0.0, color='black', linewidth=1)
ax.set_title('Sweep diagnostics summary (Spearman alpha vs test acc)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1,2, figsize=(12,4))
axes[0].scatter(aggregated_results_df['alpha_primary_mean'], aggregated_results_df['valid_accuracy_mean'], alpha=0.5, label='all')
axes[0].scatter(pareto_alpha_df['alpha_primary_mean'], pareto_alpha_df['valid_accuracy_mean'], color='darkorange', label='pareto')
axes[0].axvline(2.0, color='red', linestyle='--', linewidth=1)
axes[0].set_title('Raw W7 Pareto: alpha vs validation accuracy')
axes[0].set_xlabel('alpha_primary_mean'); axes[0].set_ylabel('valid_accuracy_mean'); axes[0].legend()

axes[1].scatter(aggregated_results_df['alpha_distance_to_2_mean'], aggregated_results_df['valid_accuracy_mean'], alpha=0.5, label='all')
axes[1].scatter(pareto_d2_df['alpha_distance_to_2_mean'], pareto_d2_df['valid_accuracy_mean'], color='green', label='pareto')
axes[1].set_title('Pareto: |alpha-2| vs validation accuracy')
axes[1].set_xlabel('alpha_distance_to_2_mean'); axes[1].set_ylabel('valid_accuracy_mean'); axes[1].legend()
plt.tight_layout(); plt.show()


# 8. Secondary Matrix Diagnostics on the Final Shortlist, W8 in particular

In [ ]:

from scipy import sparse

def build_leaf_design_matrix(model, X_input):
    d = xgb.DMatrix(X_input)
    leaves = model.predict(d, pred_leaf=True)
    if leaves.ndim == 1:
        leaves = leaves.reshape(-1, 1)
    n_samples, n_trees = leaves.shape
    rows, cols = [], []
    col_offset = 0
    for t in range(n_trees):
        ids, inv = np.unique(leaves[:, t], return_inverse=True)
        rows.extend(np.arange(n_samples))
        cols.extend(col_offset + inv)
        col_offset += len(ids)
    data = np.ones(len(rows), dtype=np.float32)
    Z = sparse.csr_matrix((data, (np.asarray(rows), np.asarray(cols))), shape=(n_samples, col_offset), dtype=np.float32)
    return Z

def alpha_from_singular_values(svals):
    s = np.asarray(svals)
    s = s[np.isfinite(s) & (s > 1e-12)]
    if s.size < 10:
        return np.nan
    evals = np.sort((s ** 2))[::-1]
    tail = evals[:max(10, int(0.3 * len(evals)))]
    xmin = np.min(tail)
    if xmin <= 0:
        return np.nan
    return float(1.0 + len(tail) / np.sum(np.log(tail / xmin)))

def secondary_metrics_for_params(params, seed=RNG):
    model, metrics, train_params = train_single_config(params, seed)
    out = {}
    try:
        Z = build_leaf_design_matrix(model, X_train_valid)
        col_support = np.asarray(Z.sum(axis=0)).ravel()
        scale = 1.0 / np.sqrt(col_support + 1e-6)
        Zn = Z @ sparse.diags(scale)
        k = min(120, min(Zn.shape) - 1) if min(Zn.shape) > 2 else 0
        if k > 2:
            from scipy.sparse.linalg import svds
            svals = svds(Zn, k=k, return_singular_vectors=False)
            out['alpha_w7_colnorm'] = alpha_from_singular_values(np.sort(np.abs(svals))[::-1])
        else:
            out['alpha_w7_colnorm'] = np.nan
    except Exception:
        out['alpha_w7_colnorm'] = np.nan

    try:
        converted_w8 = convert(
            model=model, data=X_train_valid, labels=y_train_valid,
            W='W8', nfolds=5, t_points=40, random_state=RNG,
            train_params=train_params, num_boost_round=metrics['best_round'],
            multiclass='error', return_type='torch', verbose=False,
        )
        d8 = ww.WeightWatcher(model=converted_w8).analyze(**WW_ANALYZE_OPTIONS)
        out['alpha_w8'] = extract_weightwatcher_metrics(d8, 'W8')['alpha_primary']
    except Exception:
        out['alpha_w8'] = np.nan
    return out

shortlist_ids = sorted(set([
    ACCURACY_BASELINE['configuration_id'],
    LOW_ALPHA_COMPETITIVE_ANCHOR['configuration_id'],
    best_raw_w7_accuracy_model['configuration_id'],
    best_raw_w7_low_alpha_model['configuration_id'],
    best_raw_w7_pareto_model['configuration_id'],
]))
shortlist_df = aggregated_results_df[aggregated_results_df['configuration_id'].isin(shortlist_ids)].copy()

secondary_rows = []
if not RAW_W7_ALPHA2_REACHABLE:
    for _, r in shortlist_df.iterrows():
        params = {k:(int(r[k]) if k in ['max_depth','n_estimators'] else float(r[k])) for k in PARAM_COLS}
        sec = secondary_metrics_for_params(params)
        secondary_rows.append({'configuration_id': r['configuration_id'], **sec})

secondary_df = pd.DataFrame(secondary_rows) if secondary_rows else pd.DataFrame(columns=['configuration_id','alpha_w7_colnorm','alpha_w8'])
if not secondary_df.empty:
    shortlist_df = shortlist_df.merge(secondary_df, on='configuration_id', how='left')
else:
    shortlist_df['alpha_w7_colnorm'] = np.nan
    shortlist_df['alpha_w8'] = np.nan


# 9. Final Model Selection

In [ ]:

best_validation_model = aggregated_results_df.sort_values('valid_accuracy_mean', ascending=False).iloc[0]

final_flags = {
    'best_validation_model': best_validation_model['configuration_id'],
    'best_raw_w7_low_alpha_model': best_raw_w7_low_alpha_model['configuration_id'],
    'best_raw_w7_pareto_model': best_raw_w7_pareto_model['configuration_id'],
}

best_raw_w7_alpha2_model = None
if RAW_W7_ALPHA2_REACHABLE:
    alpha2_band = aggregated_results_df[(aggregated_results_df['alpha_primary_mean'] >= 1.85) & (aggregated_results_df['alpha_primary_mean'] <= 2.15)]
    if not alpha2_band.empty:
        best_raw_w7_alpha2_model = alpha2_band.sort_values('valid_accuracy_mean', ascending=False).iloc[0]

best_secondary_alpha2_model = None
if (not RAW_W7_ALPHA2_REACHABLE) and (not shortlist_df.empty):
    sec_pool = shortlist_df.copy()
    sec_pool['secondary_alpha'] = sec_pool['alpha_w8'].fillna(sec_pool['alpha_w7_colnorm'])
    sec_pool = sec_pool[sec_pool['secondary_alpha'].notna()].copy()
    if not sec_pool.empty:
        in_band = sec_pool[(sec_pool['secondary_alpha'] >= 1.85) & (sec_pool['secondary_alpha'] <= 2.15)]
        if not in_band.empty:
            best_secondary_alpha2_model = in_band.sort_values('valid_accuracy_mean', ascending=False).iloc[0]
        else:
            sec_pool['secondary_alpha_distance_to_2'] = (sec_pool['secondary_alpha'] - 2.0).abs()
            best_secondary_alpha2_model = sec_pool.sort_values(['secondary_alpha_distance_to_2','valid_accuracy_mean'], ascending=[True,False]).iloc[0]

shortlist_df['is_accuracy_baseline'] = shortlist_df['configuration_id'].eq(ACCURACY_BASELINE['configuration_id'])
shortlist_df['is_low_alpha_competitive_anchor'] = shortlist_df['configuration_id'].eq(LOW_ALPHA_COMPETITIVE_ANCHOR['configuration_id'])
shortlist_df['is_best_validation_model'] = shortlist_df['configuration_id'].eq(best_validation_model['configuration_id'])
shortlist_df['is_best_raw_w7_low_alpha_model'] = shortlist_df['configuration_id'].eq(best_raw_w7_low_alpha_model['configuration_id'])
shortlist_df['is_best_raw_w7_pareto_model'] = shortlist_df['configuration_id'].eq(best_raw_w7_pareto_model['configuration_id'])
shortlist_df['is_best_raw_w7_alpha2_model'] = False if best_raw_w7_alpha2_model is None else shortlist_df['configuration_id'].eq(best_raw_w7_alpha2_model['configuration_id'])
shortlist_df['is_best_secondary_alpha2_model'] = False if best_secondary_alpha2_model is None else shortlist_df['configuration_id'].eq(best_secondary_alpha2_model['configuration_id'])

display(shortlist_df.sort_values('valid_accuracy_mean', ascending=False))


In [ ]:

# Final shortlist and secondary-matrix visualizations
label_points = {
    'ACCURACY_BASELINE': ACCURACY_BASELINE['configuration_id'],
    'LOW_ALPHA_COMPETITIVE_ANCHOR': LOW_ALPHA_COMPETITIVE_ANCHOR['configuration_id'],
    'best_validation_model': best_validation_model['configuration_id'],
    'best_raw_w7_low_alpha_model': best_raw_w7_low_alpha_model['configuration_id'],
    'best_raw_w7_pareto_model': best_raw_w7_pareto_model['configuration_id'],
}
if best_secondary_alpha2_model is not None:
    label_points['best_secondary_alpha2_model'] = best_secondary_alpha2_model['configuration_id']

plt.figure(figsize=(8,5))
plt.scatter(aggregated_results_df['alpha_primary_mean'], aggregated_results_df['valid_accuracy_mean'], alpha=0.25, s=30)
for lbl,cid in label_points.items():
    r = aggregated_results_df[aggregated_results_df['configuration_id']==cid].iloc[0]
    plt.scatter([r['alpha_primary_mean']],[r['valid_accuracy_mean']], s=120, marker='*')
    plt.annotate(lbl, (r['alpha_primary_mean'], r['valid_accuracy_mean']), fontsize=8)
plt.axvline(2.0, color='red', linestyle='--', linewidth=1)
plt.xlabel('alpha_primary_mean'); plt.ylabel('valid_accuracy_mean')
plt.title('Final named models on raw W7 alpha/validation plane')
plt.tight_layout(); plt.show()

if shortlist_df[['alpha_w7_colnorm','alpha_w8']].notna().any().any():
    tmp = shortlist_df.melt(
        id_vars=['configuration_id','valid_accuracy_mean'],
        value_vars=['alpha_primary_mean','alpha_w7_colnorm','alpha_w8'],
        var_name='matrix_variant', value_name='alpha_value'
    )
    tmp = tmp[tmp['alpha_value'].notna()]
    plt.figure(figsize=(8,4))
    sns.scatterplot(data=tmp, x='alpha_value', y='valid_accuracy_mean', hue='matrix_variant', style='configuration_id', s=100)
    plt.axvline(2.0, color='red', linestyle='--', linewidth=1)
    plt.title('Secondary-matrix alpha diagnostics on shortlist')
    plt.tight_layout(); plt.show()


### What counts as success now

Success is defined as at least one of:
1. competitive raw-W7 model with materially lower alpha than the original baseline,
2. a clear raw-W7 Pareto frontier with explicit low-alpha/high-accuracy tradeoff,
3. evidence that a secondary matrix variant (W7-column-normalized or W8) yields alpha near 2 on the final shortlist even when raw W7 does not.


# 10. Conclusion

This repaired workflow separates diagnosis from optimization: sweep-local diagnostics identify helpful manifolds first, targeted refinement explores only those manifolds, and final selection reports raw-W7 accuracy, raw-W7 low-alpha frontier, and secondary-matrix alpha≈2 diagnostics separately.


In [ ]:

# Save required result tables
results_root = Path('results')
results_root.mkdir(parents=True, exist_ok=True)

per_seed_out = results_root / 'spambase_per_seed_results.csv'
agg_out = results_root / 'spambase_aggregated_results.csv'
diag_out = results_root / 'spambase_sweep_diagnostics.csv'
shortlist_out = results_root / 'spambase_final_shortlist.csv'

per_seed_results_df.to_csv(per_seed_out, index=False)
aggregated_results_df.to_csv(agg_out, index=False)
sweep_diagnostics_df.to_csv(diag_out, index=False)
shortlist_df.to_csv(shortlist_out, index=False)

print('Saved:', per_seed_out, agg_out, diag_out, shortlist_out)
